In [38]:
import pandas as pd
import glob

json_files = [r'../Data/processed/creator_cooked_xh.json',r'../DataPreprocessing\score1_nested.json',r'../DataPreprocessing\dim2_score_nested.json',
              r'../DataPreprocessing\dim3_score_nested.json',r'../DataPreprocessing\dim4_score_nested.json',r'../DataPreprocessing\dim5_score_nested.json']
data_frames = [pd.read_json(file) for file in json_files]
df_merge = data_frames[0]
for i in range(1,len(data_frames)):
    df_merge = pd.merge(
    df_merge,
    data_frames[i],
    on='user_id',
    suffixes=('', f'_dim_{i}')
    )
df_merge.drop(columns=['score_2'], inplace=True)
df_merge.columns


Index(['user_id', 'nickname', 'avatar', 'desc', 'ip_location', 'follows',
       'fans', 'interaction', 'last_modify_ts', 'pic_per_normal_note',
       'video_ratio', 'hot_note_count', 'total_share_counts_hot_ratio',
       'last_note2now', 'last_hot_note2now', 'weighted_total_share_counts',
       'liked_count', 'collected_count', 'comment_count', 'share_count',
       'note_count', 'location', 'ff_ratio', 'age_koc', 'is_female', 'min',
       'max', 'post_span', 'first_post_time', 'account_length', 'history_avg',
       'history_std', 'post_avg', 'post_std', 'liked_90', 'collected_90',
       'comment_90', 'share_90', 'note_count_90', 'liked_180', 'collected_180',
       'comment_180', 'share_180', 'note_count_180', '图文报价(RMB)', '图文报价有对号',
       '平台等级', '活跃粉丝占比(仅>1K)', '粉丝女性比例', '粉丝年龄<18', '粉丝年龄18-24', '粉丝年龄25-34',
       '粉丝年龄35-44', '粉丝年龄>44', '兴趣标签', '地域分布', 'score1_account_influence',
       'component_scores', 'score2_content_media', 'component_scores_dim_2',
       'score3_con

In [39]:
import numpy as np
# 4. Specify columns to round and perform rounding
nested_col = ['component_scores', 'component_scores_dim_2', 'component_scores_dim_3', 'component_scores_dim_4',
       'component_scores_dim_5']
columns_to_round = ['score1_account_influence',
       'score2_content_media',
       'score3_content_quality',
       'score4_target_audience_match', 'score5_business_coop']  # Update with your column names
df_merge[columns_to_round] = df_merge[columns_to_round].round(1)
weights = {
    'score1_account_influence': 0.30,
    'score2_content_media': 0.20,
    'score3_content_quality': 0.20,
    'score4_target_audience_match': 0.20,
    'score5_business_coop': 0.10
}

# 验证列名与权重键的一致性
assert set(columns_to_round) == set(weights.keys()), "列名与权重键不匹配"

# 加权求和
df_merge['total_score'] = (
    df_merge[columns_to_round]          # 选择要计算的列
    .multiply(pd.Series(weights))       # 每列乘对应权重
    .sum(axis=1)).round(1)
df_merge = df_merge.rename(columns={
    'desc': 'profile_intro'
})
for i in range(len(df_merge)):
      rows = df_merge.iloc[i]
      #rows.loc['total_score'] = rows['score1_account_influence']+rows['score2_content_media']+rows['score3_content_quality']+rows['score4_target_audience_match']+rows['score5_business_coop']
      for j in nested_col:
            #print(rows.loc[j])
            for z in rows.loc[j]:
                  #print(rows.loc[j][z])
                  rows.loc[j][z]=np.round(rows.loc[j][z],1)

    

df_merge.to_json('creator_final.json',orient='records', indent=4, force_ascii=False)
